In [ ]:
import torch
import numpy as np
import math
import random
from scigen.pl_modules.diffusion_w_type import MAX_ATOMIC_NUM 
Pi = math.pi

### Technical Pathway 1

Extend SC_Base in SCIGEN-main/script/sc_utils.py

In [ ]:
class SC_Nanotube(SC_Base):
    """
    Base class for 1D nanotube structures
    Nanotubes have periodicity along z-axis (axial)
    and circular/helical symmetry in xy-plane
    """
    def __init__(self, bond_len, num_atom, type_known, 
                 nanotube_radius, helicity, frac_z, 
                 c_vec_cons, reduced_mask, device):
        # Nanotube-specific parameters
        self.nanotube_radius = nanotube_radius  # Angstroms
        self.helicity = helicity  # (n, m) for chiral angle
        self.chiral_angle = self._compute_chiral_angle()
        
        super().__init__(bond_len, num_atom, type_known, 
                         frac_z, c_vec_cons, reduced_mask, device)
    
    def _compute_chiral_angle(self):
        """Calculate chiral angle from (n,m) indices"""
        n, m = self.helicity
        return math.atan2(math.sqrt(3)*m, 2*n + m)
    
    def _generate_nanotube_coords(self):
        """
        Generate atomic positions on nanotube surface
        - Position on circumference (angle θ)
        - Position along axis (z)
        """
        n_atoms_circumference = int(2 * math.pi * self.nanotube_radius / self.bond_len)
        
        frac_coords = []
        for z_idx in range(self.num_known):
            for theta_idx in range(n_atoms_circumference):
                theta = 2 * math.pi * theta_idx / n_atoms_circumference
                x = self.nanotube_radius * math.cos(theta)
                y = self.nanotube_radius * math.sin(theta)
                z = self.frac_z + (z_idx * self.bond_len)
                
                frac_coords.append([x, y, z])
        
        return torch.tensor(frac_coords, dtype=torch.float)
    
    def get_cell(self):
        """
        For nanotubes: define unit cell along z-axis
        xy-plane is non-periodic (boundary conditions)
        """
        # Lattice parameters for nanotube (minimal periodicity)
        self.cell_lengths = torch.tensor(
            [2*self.nanotube_radius, 
             2*self.nanotube_radius, 
             self.bond_len],  # Periodic along z
            dtype=torch.float, 
            device=self.device
        )
        self.cell_angles_d = torch.tensor([90, 90, 90], dtype=torch.float, device=self.device)
        
        return lattice_params_to_matrix_xy_torch(
            self.cell_lengths.unsqueeze(0), 
            self.cell_angles_d.unsqueeze(0)
        ).squeeze(0)
    
    def get_mask_l(self):
        """
        Lattice mask for nanotubes:
        - z-direction (axial) is ALWAYS periodic → constrained
        - xy-plane diameter can vary → free to denoise
        """
        return torch.tensor([[[1, 1, 0],  # x, y components constrained
                            [1, 1, 0], 
                            [0, 0, 1]]])  # z component free


In [ ]:
class SC_CNNanotube(SC_Nanotube):
    """Carbon nanotube (armchair, zigzag, or chiral)"""
    def __init__(self, bond_len, num_atom, type_known, 
                 helicity=(10, 10), frac_z=0.5, 
                 c_vec_cons={'scale': None, 'vert': False},
                 reduced_mask=True, device='cpu'):
        self.nanotube_radius = self._compute_radius_from_helicity(helicity)
        super().__init__(bond_len, num_atom, type_known, 
                        self.nanotube_radius, helicity, frac_z, 
                        c_vec_cons, reduced_mask, device)
        
        self.a_scale, self.b_scale = 1, 1  # Normalized
        self.cell = self.get_cell()
        self.frac_known = self._generate_nanotube_coords()
        self.num_known = self.frac_known.shape[0]
    
    @staticmethod
    def _compute_radius_from_helicity(helicity):
        n, m = helicity
        a_cc = 1.42  # C-C bond length (Angstroms)
        return a_cc * math.sqrt(3 * (n**2 + m**2 + n*m)) / (2 * math.pi)

class SC_BNNanotube(SC_Nanotube):
    """Boron Nitride nanotube"""
    def __init__(self, bond_len, num_atom, type_known, 
                 helicity=(10, 10), frac_z=0.5,
                 c_vec_cons={'scale': None, 'vert': False},
                 reduced_mask=True, device='cpu'):
        # Similar to CNNanotube but with BN bond length
        a_bn = 1.45  # B-N bond length
        self.nanotube_radius = a_bn * math.sqrt(3 * (helicity[0]**2 + helicity[1]**2 + 
                                                       helicity[0]*helicity[1])) / (2 * math.pi)
        super().__init__(bond_len, num_atom, type_known, 
                        self.nanotube_radius, helicity, frac_z,
                        c_vec_cons, reduced_mask, device)
        
        self.a_scale, self.b_scale = 1, 1
        self.cell = self.get_cell()
        self.frac_known = self._generate_nanotube_coords()
        self.num_known = self.frac_known.shape[0]

# Register in sc_dict
sc_dict.update({
    'cnt': SC_CNNanotube,
    'bnt': SC_BNNanotube,
})


### Tehcnical Pathway 2: Database Integration

Modify SCIGEN-main/script/gen_utils.py

In [ ]:
import pickle as pkl
from pathlib import Path

In [ ]:
class NanotubeDatabase:
    """Load and sample from nanotube structure database"""
    
    def __init__(self, db_path):
        self.db_path = Path(db_path)
        self.data = self._load_database()
    
    def _load_database(self):
        """Load nanotube structures from database"""
        if self.db_path.suffix == '.pkl':
            with open(self.db_path, 'rb') as f:
                return pkl.load(f)  # Expected: list of dicts with keys:
                                    # 'frac_coords', 'lattice', 'atom_types', 'helicity'
        elif self.db_path.suffix == '.cif':
            return self._load_from_cif_dir(self.db_path)
    
    def _load_from_cif_dir(self, cif_dir):
        """Load multiple .cif files into standardized format"""
        structures = []
        for cif_file in Path(cif_dir).glob('*.cif'):
            # Use pymatgen or ASE to parse
            structure = self._parse_cif(cif_file)
            structures.append(structure)
        return structures
    
    def get_random_nanotube(self):
        """Sample random nanotube from database"""
        return random.choice(self.data)
    
    def get_by_helicity(self, n, m):
        """Get nanotube with specific (n,m) helicity"""
        matches = [nt for nt in self.data 
                  if nt['helicity'] == (n, m)]
        return random.choice(matches) if matches else None

class NanotubeDataset(SampleDataset):
    """Extended dataset for 1D nanotubes"""
    
    def __init__(self, nanotube_db_path, num_samples, 
                 helicity_list=None, **kwargs):
        self.nt_db = NanotubeDatabase(nanotube_db_path)
        self.helicity_list = helicity_list or [
            (10, 10), (12, 0), (8, 8), (6, 6)  # Sample helicities
        ]
        
        # Initialize parent but override sc_list
        super().__init__(**kwargs)
        self.nanotube_mode = True
        
    def process(self):
        """Override to use database nanotubes"""
        self.data_list = []
        
        for i in range(self.total_num):
            # Sample helicity
            helicity = random.choice(self.helicity_list)
            
            # Get from database or generate
            nt_structure = self.nt_db.get_by_helicity(*helicity)
            if nt_structure is None:
                # Generate if not in database
                sc_obj = sc_dict['cnt']
                nt_obj = sc_obj(
                    bond_len=random.uniform(1.4, 1.5),
                    num_atom=None,
                    type_known=random.choice(self.known_species),
                    helicity=helicity,
                    frac_z=random.uniform(0, 1),
                    c_vec_cons=self.c_vec_cons,
                    reduced_mask=self.reduced_mask,
                    device=self.device
                )
                nt_structure = {
                    'frac_coords': nt_obj.frac_coords,
                    'lattice': nt_obj.cell,
                    'atom_types': nt_obj.atom_types,
                    'mask_x': nt_obj.mask_x,
                    'mask_l': nt_obj.mask_l,
                    'mask_t': nt_obj.mask_t,
                }
            
            self.data_list.append(nt_structure)
